In [1]:
pip install opencv-python

Defaulting to user installation because normal site-packages is not writeable
  Using cached numpy-2.5.1-cp312-cp312-win_amd64.whl.metadata (6.6 kB)
Using cached numpy-2.5.1-cp312-cp312-win_amd64.whl (12.4 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gensim 4.3.3 requires numpy<2.0,>=1.18.5, but you have numpy 2.5.1 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.5.1 which is incompatible.
scipy 1.13.1 requires numpy<2.3,>=1.22.4, but you have numpy 2.5.1 which is incompatible.
streamlit 1.37.1 requires rich<14,>=10.14.0, but you have rich 15.0.0 which is incompatible.


In [2]:
pip install ultralytics

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install numpy==1.26.4

Defaulting to user installation because normal site-packages is not writeable
  Using cached numpy-1.26.4-cp312-cp312-win_amd64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-win_amd64.whl (15.5 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.5.1
    Uninstalling numpy-2.5.1:
      Successfully uninstalled numpy-2.5.1
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
streamlit 1.37.1 requires rich<14,>=10.14.0, but you have rich 15.0.0 which is incompatible.


In [4]:
import numpy as np
print(np.__version__)

import cv2
from ultralytics import YOLO

print("Everything works!")

1.26.4
Everything works!


In [ ]:
import cv2
import torch
from ultralytics import YOLO

# Load model
model = YOLO(r"D:\Carrer\EM_ML_AI\Final Project 3\YOLO26n_best.pt")

# Check GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Open camera
cap = cv2.VideoCapture(0)

# Reduce camera resolution for faster processing
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

if not cap.isOpened():
    print("❌ Could not open camera")
    exit()

print("✅ Camera started!")
print("Press 'q' to quit")

while True:

    ret, frame = cap.read()

    if not ret:
        print("❌ Failed to read frame")
        break

    # YOLO inference
    results = model.predict(
        source=frame,
        conf=0.30,
        imgsz=320,       # Smaller image = faster inference
        device=device,
        verbose=False
    )

    # Draw results
    annotated_frame = results[0].plot()

    # Display
    cv2.imshow(
        "YOLO Real-Time Instance Segmentation",
        annotated_frame
    )

    # Press Q to quit
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

Using device: cuda
✅ Camera started!
Press 'q' to quit


In [ ]:
import cv2
from ultralytics import RTDETR

MODEL_PATH = r"D:\Carrer\EM_ML_AI\Final Project 3\rt-deter_best.pt" 
try:
    model = RTDETR(MODEL_PATH)
except Exception as e:
    print(f"Error loading model: {e}")
    exit()

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Cannot open the camera!")
    exit()

while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = model.predict(frame, conf=0.50, verbose=False)
    annotated_frame = results[0].plot()
    
    inference_time = results[0].speed.get('inference', 0)
    fps = 1000 / inference_time if inference_time > 0 else 0
    num_detections = len(results[0].boxes)
    
    cv2.putText(
        annotated_frame, 
        f"RT-DETR | FPS: {fps:.1f} | Objects: {num_detections}", 
        (10, 30), 
        cv2.FONT_HERSHEY_SIMPLEX, 
        0.7, 
        (0, 0, 255), 
        2
    )

    cv2.imshow("Live RT-DETR Detection", annotated_frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [5]:
import cv2
import numpy as np
import torch
from ultralytics import YOLO, RTDETR

# ---- Load both models once ----
YOLO_PATH = r"D:\Carrer\EM_ML_AI\Final Project 3\YOLO26n_best.pt"
RTDETR_PATH = r"D:\Carrer\EM_ML_AI\Final Project 3\rt-deter_best.pt"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

yolo_model = YOLO(YOLO_PATH)
rtdetr_model = RTDETR(RTDETR_PATH)

# ---- Open camera ----
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

if not cap.isOpened():
    print("❌ Cannot open the camera!")
    exit()

print("✅ Camera started! Press 'q' to quit.")

def run_and_annotate(model, frame, label, color):
    results = model.predict(frame, conf=0.5, imgsz=320, device=device, verbose=False)
    annotated = results[0].plot()

    inference_time = results[0].speed.get('inference', 0)
    fps = 1000 / inference_time if inference_time > 0 else 0
    num_detections = len(results[0].boxes)

    cv2.putText(
        annotated,
        f"{label} | FPS: {fps:.1f} | Obj: {num_detections}",
        (10, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        color,
        2
    )
    return annotated

while True:
    ret, frame = cap.read()
    if not ret:
        print("❌ Failed to read frame")
        break

    # Run YOLO
    yolo_annotated = run_and_annotate(yolo_model, frame, "YOLO", (0, 255, 0))

    # Run RT-DETR
    rtdetr_annotated = run_and_annotate(rtdetr_model, frame, "RT-DETR", (0, 0, 255))

    # Resize both to same height (in case models output different sizes) then stack side by side
    h = 480
    yolo_resized = cv2.resize(yolo_annotated, (640, h))
    rtdetr_resized = cv2.resize(rtdetr_annotated, (640, h))

    combined = np.hstack((yolo_resized, rtdetr_resized))

    cv2.imshow("YOLO (Left)  |  RT-DETR (Right)", combined)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Using device: cuda
✅ Camera started! Press 'q' to quit.


In [5]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
Quadro T2000
